# Names Are Not Boxes
One of the biggest turning points in Python understanding comes when names stop looking like containers and start looking like labels attached to objects. Once that clicks, aliasing, mutation, copying, and function calls all become much easier to reason about.

When people struggle with **Names Are Not Boxes**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Replace the box model with the reference model
- Understand aliasing clearly
- See why assignment does not imply copying
- Prepare for function arguments and mutation rules


When you bind two names to the same list, both names carry references to one shared list object. That one object lives in memory once, but several names may lead to it. That is why one change can appear through multiple names.


In [1]:
items = [1, 2, 3]
alias = items
print(id(items), id(alias))
alias.append(4)
print(items)
print(alias)


2250321928768 2250321928768
[1, 2, 3, 4]
[1, 2, 3, 4]


Even assignment bytecode is revealing. Python evaluates the right side, then stores the resulting object reference under the target name. It is a store operation, not an implicit clone operation.


In [2]:
import dis

def bind():
    a = [1, 2]
    b = a
    return a, b

dis.dis(bind)


  3           0 RESUME                   0

  4           2 LOAD_CONST               1 (1)
              4 LOAD_CONST               2 (2)
              6 BUILD_LIST               2
              8 STORE_FAST               0 (a)

  5          10 LOAD_FAST                0 (a)
             12 STORE_FAST               1 (b)

  6          14 LOAD_FAST                0 (a)
             16 LOAD_FAST                1 (b)
             18 BUILD_TUPLE              2
             20 RETURN_VALUE


The central event is binding, not copying.

Two or more names can point to the same object and that is not a bug by itself.

If an object changes in place, every reference to that object observes the change.

When a name is rebound, the old object may continue to exist if something else still refers to it.


The list is shared, so a mutation through one name is visible through the other.


In [3]:
left = [10, 20]
right = left
right.append(30)
print(left)
print(right)


[10, 20, 30]
[10, 20, 30]


Here `left` is moved to a new list, so `right` still refers to the earlier one.


In [4]:
left = [10, 20]
right = left
left = left + [30]
print(left)
print(right)


[10, 20, 30]
[10, 20]


The parameter name inside the function becomes another reference to the passed object.


In [5]:
def add_value(bucket):
    bucket.append(99)

nums = [1, 2]
add_value(nums)
print(nums)


[1, 2, 99]


This is not only a classroom topic. **Names Are Not Boxes** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Saying “Python passed it by reference” without carefully explaining what that means
- Expecting `a = b` to make a full copy
- Confusing rebinding with mutation


- What is aliasing?
- Why does changing a list inside a function affect the caller sometimes?
- What really happens during assignment?


- Names are references, not boxes.
- Aliasing is normal.
- Mutation and rebinding are different events.


If this notebook did its job, **Names Are Not Boxes** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Names Are Not Boxes is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Names Are Not Boxes, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x0000020BF1867D00, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_29820\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST        

A very deep question here is whether the operation changed a name, changed an object, or changed both. Many bugs become obvious as soon as you answer that correctly. Python is relentlessly consistent about this. Names are rebound. Mutable objects are changed in place. Immutable results usually mean a new object was produced and then bound to some name.

That is why nested structures deserve extra care. Two outer containers may be different objects while still sharing one important inner object.


In [7]:
inner = {'count': 0}
left = [inner]
right = [inner]
print('outer ids:', id(left), id(right))
print('shared inner ids:', id(left[0]), id(right[0]))
left[0]['count'] += 1
print(left)
print(right)


outer ids: 2250321934208 2250321933312
shared inner ids: 2250321876800 2250321876800
[{'count': 1}]
[{'count': 1}]


The deepest benefit of this chapter is that it changes how you talk to yourself while reading code. Instead of saying ?this variable contains that value?, you start saying ?this name points to that object?. That small change makes aliasing, mutation, rebinding, parameter passing, and copying much easier to reason about. It also helps you avoid a lot of accidental bugs in data processing and object-oriented code.


## Final Takeaway

The deepest way to revise **Names Are Not Boxes** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
